In [51]:
# ============================================================
# CELL 1: IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import re

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report


In [52]:
# ============================================================
# THEORY:
# Libraries help in:
# - Data handling (pandas)
# - Text processing (re)
# - Machine learning (sklearn)
# ============================================================



# ============================================================
# CELL 2: LOAD DATASET
# ============================================================

df = pd.read_csv("Train.csv")

# Remove extra spaces in column names
df.columns = df.columns.str.strip()

print("Dataset Shape:", df.shape)
print("Columns:", df.columns)
print(df.head())


Dataset Shape: (1986, 2)
Columns: Index(['text', 'label'], dtype='object')
                                                text          label
0  ENGINEERING CONSULTANT Professional Summary To...     CONSULTANT
1  CARPENTER Summary Carpenter Foreman Position w...   CONSTRUCTION
2  DIGITAL MARKETING SPECIALIST Summary Digital m...  DIGITAL-MEDIA
3  SOUS CHEF Work Experience Sous Chef Jul 2010 C...           CHEF
4  DONOR ADVOCATE Professional Summary Organized ...       ADVOCATE


In [53]:
# ============================================================
# THEORY:
# Dataset contains:
# - Text data (resume)
# - Label (job category)
# ============================================================



# ============================================================
# CELL 3: HANDLE COLUMN NAMES (AUTO FIX)
# ============================================================

# Automatically detect columns
text_column = df.columns[0]
label_column = df.columns[1]

print("Using Text Column:", text_column)
print("Using Label Column:", label_column)


Using Text Column: text
Using Label Column: label


In [54]:
# ============================================================
# THEORY:
# Ensures code works even if dataset column names differ
# ============================================================



# ============================================================
# CELL 4: TEXT PREPROCESSING
# ============================================================

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

df['clean_text'] = df[text_column].apply(clean_text)

print("\nCleaned Text Sample:")
print(df[[text_column, 'clean_text']].head())



Cleaned Text Sample:
                                                text  \
0  ENGINEERING CONSULTANT Professional Summary To...   
1  CARPENTER Summary Carpenter Foreman Position w...   
2  DIGITAL MARKETING SPECIALIST Summary Digital m...   
3  SOUS CHEF Work Experience Sous Chef Jul 2010 C...   
4  DONOR ADVOCATE Professional Summary Organized ...   

                                          clean_text  
0  engineering consultant professional summary to...  
1  carpenter summary carpenter foreman position w...  
2  digital marketing specialist summary digital m...  
3  sous chef work experience sous chef jul  compa...  
4  donor advocate professional summary organized ...  


In [55]:
# ============================================================
# THEORY:
# - Lowercasing
# - Removing special characters
# Improves model performance
# ============================================================



# ============================================================
# CELL 5: FEATURE EXTRACTION (TF-IDF + N-GRAMS)
# ============================================================

tfidf = TfidfVectorizer(
    max_features=20000,        # use more features for large data
    ngram_range=(1,2),         # unigram + bigram 🔥
    stop_words='english'       # remove common words
)

X = tfidf.fit_transform(df['clean_text'])
y = df[label_column]

print("Feature Matrix Shape:", X.shape)


Feature Matrix Shape: (1986, 20000)


In [56]:
# ============================================================
# THEORY:
# TF-IDF converts text → numbers
# N-grams capture word combinations like:
# "machine learning", "data science"
# ============================================================



# ============================================================
# CELL 6: TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # 🔥 important for balanced training
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])



Training samples: 1588
Testing samples: 398


In [57]:
# ============================================================
# THEORY:
# Stratified split ensures:
# Each class is equally represented
# ============================================================



# ============================================================
# CELL 7: LOGISTIC REGRESSION MODEL
# ============================================================

model = LogisticRegression(
    max_iter=10000,             # more iterations for large data
    class_weight='balanced'    # handle class imbalance 🔥
)

model.fit(X_train, y_train)

print("✅ Model Training Completed!")


✅ Model Training Completed!


In [58]:
# ============================================================
# THEORY:
# Logistic Regression:
# - Classification algorithm
# - Predicts probability of each class
# - Works well with high-dimensional text data
# ============================================================



# ============================================================
# CELL 8: PREDICTIONS
# ============================================================

y_pred = model.predict(X_test)

print("Sample Predictions:", y_pred[:10])


Sample Predictions: ['AVIATION' 'INFORMATION-TECHNOLOGY' 'HEALTHCARE' 'CONSTRUCTION'
 'PUBLIC-RELATIONS' 'SALES' 'ADVOCATE' 'AGRICULTURE' 'DIGITAL-MEDIA'
 'HEALTHCARE']


In [59]:
# ============================================================
# THEORY:
# Model predicts category based on learned patterns
# ============================================================



# ============================================================
# CELL 9: MODEL EVALUATION
# ============================================================

accuracy = accuracy_score(y_test, y_pred)

print("\n✅ Accuracy:", accuracy)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))



✅ Accuracy: 0.6457286432160804

Classification Report:

                        precision    recall  f1-score   support

            ACCOUNTANT       0.74      0.94      0.83        18
              ADVOCATE       0.44      0.61      0.51        18
           AGRICULTURE       0.60      0.30      0.40        10
               APPAREL       0.75      0.35      0.48        17
                  ARTS       0.67      0.25      0.36        16
            AUTOMOBILE       0.00      0.00      0.00         6
              AVIATION       0.65      0.68      0.67        19
               BANKING       0.90      0.50      0.64        18
                   BPO       0.00      0.00      0.00         3
  BUSINESS-DEVELOPMENT       0.50      0.50      0.50        18
                  CHEF       0.74      0.85      0.79        20
          CONSTRUCTION       0.83      0.79      0.81        19
            CONSULTANT       0.60      0.17      0.26        18
              DESIGNER       0.88      0.82   

In [60]:
# ============================================================
# THEORY:
# Accuracy → overall performance
# Precision → correctness
# Recall → coverage
# F1-score → balance of both
# ============================================================



# ============================================================
# CELL 10: TEST WITH NEW RESUME
# ============================================================

new_resume = [
    "Experienced web developer skilled in HTML, CSS, JavaScript and React"
]

# Clean
new_clean = [clean_text(text) for text in new_resume]

# Vectorize
new_vector = tfidf.transform(new_clean)

# Predict
prediction = model.predict(new_vector)

print("\nPredicted Category:", prediction[0])


# ============================================================
# THEORY:
# Same preprocessing + TF-IDF must be applied to new data
# ============================================================


Predicted Category: CONSULTANT
